# Verification of the LWR / Godunov scheme (PDE part)

Single-road checks that the finite-volume Godunov scheme reproduces **exact** solutions of the
LWR conservation law, using the **same** Greenshields fundamental diagram and demand--supply
(Godunov) flux as the city model (`traffic_model_full_en4.ipynb`):

$$q(\rho)=v_f\,\rho\left(1-\tfrac{\rho}{\rho_{jam}}\right),\quad \rho_c=\tfrac{\rho_{jam}}2,\quad Q_{max}=\tfrac14 v_f\rho_{jam}.$$

1. **Red light** → backward **shock**, compared to the analytical **Rankine--Hugoniot** speed
   $s=\dfrac{q(\rho_R)-q(\rho_L)}{\rho_R-\rho_L}$.
2. **Green light** → **rarefaction** fan.
3. A global **mass ledger** confirms conservation to machine precision.

Saves `shock_spacetime.png`, `shock_trajectory.png`, `shock_profiles.png`, `rarefaction_profiles.png`.

In [ ]:
import numpy as np, matplotlib.pyplot as plt

VF   = 50/3.6          # free-flow speed (m/s) = 50 km/h  (same class-default family as the model)
RJAM = 0.15            # jam density (veh/m), one lane    (LANE_JAM in the model)
RC   = RJAM/2
QMAX = VF*RJAM/4
CFL  = 0.9

def q(r):      return VF*r*(1-r/RJAM)                 # Greenshields flow
def demand(r): return np.where(r<=RC, q(r), QMAX)     # Delta
def supply(r): return np.where(r<=RC, QMAX, q(r))     # Sigma

def godunov_step(rho, dt, dx, lg, rg):
    r = np.concatenate(([lg], rho, [rg]))
    F = np.minimum(demand(r[:-1]), supply(r[1:]))     # face fluxes (len N+1)
    return np.clip(rho - dt/dx*(F[1:]-F[:-1]), 0.0, RJAM), F

# ---------------- SHOCK (red light): light upstream meets a stopped queue ----------------
L, dx = 1000.0, 5.0
N  = int(L/dx); x = (np.arange(N)+0.5)*dx; x0 = 700.0
rL = 0.12*RJAM; rR = RJAM
rho = np.where(x < x0, rL, rR).astype(float)
dt = CFL*dx/VF; Tend = 300.0; steps = int(Tend/dt)
s_rh = (q(rR)-q(rL))/(rR-rL)

times, fronts, ST = [], [], []; se = max(1, steps//120); mid = 0.5*(rL+rR)
for k in range(steps+1):
    idx = np.where(rho >= mid)[0]
    if idx.size:
        j = idx[0]
        xf = x[j-1] + (mid-rho[j-1])/(rho[j]-rho[j-1])*dx if 0 < j < N and rho[j-1] < mid <= rho[j] else x[j]
        times.append(k*dt); fronts.append(xf)
    if k % se == 0: ST.append(rho.copy())
    if k < steps: rho, _ = godunov_step(rho, dt, dx, rho[0], rho[-1])
times = np.array(times); fronts = np.array(fronts)
m = (times > 30) & (times < Tend-30); s_sim = np.polyfit(times[m], fronts[m], 1)[0]
err = abs(s_sim-s_rh)/abs(s_rh)*100
print(f"[SHOCK] RH theory s = {s_rh:.4f} m/s | simulated s = {s_sim:.4f} m/s | error {err:.3f}%")

ST = np.array(ST)
fig, ax = plt.subplots(figsize=(6,4.5))
im = ax.imshow(ST, aspect="auto", origin="lower", extent=[0,L,0,len(ST)*se*dt], cmap="viridis")
tt = np.linspace(times.min(), times.max(), 50)
ax.plot(x0+s_rh*tt, tt, "c--", lw=2, label=f"RH line s={s_rh:.3f} m/s")
ax.set_xlabel("x (m)"); ax.set_ylabel("t (s)"); ax.set_title("Backward shock: density in space-time")
plt.colorbar(im, ax=ax, label=r"$\rho$ (veh/m)"); ax.legend(loc="upper right")
fig.tight_layout(); fig.savefig("shock_spacetime.png", dpi=110); plt.show()

fig, ax = plt.subplots(figsize=(6,4.5))
ax.plot(times, fronts, "k.", ms=3, label="simulation (measured front)")
ax.plot(times, x0+s_rh*times, "r-", lw=1.8, label=f"theory x0+s*t, s={s_rh:.3f}")
ax.set_xlabel("t (s)"); ax.set_ylabel("front position x (m)")
ax.set_title(f"Front trajectory (error {err:.2f}%)"); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig("shock_trajectory.png", dpi=110); plt.show()

fig, ax = plt.subplots(figsize=(6,4.5))
for fr in [0.0,0.33,0.66,1.0]:
    kk = int(fr*(len(ST)-1)); ax.plot(x, ST[kk], lw=1.6, label=f"t={kk*se*dt:.0f} s")
ax.set_xlabel("x (m)"); ax.set_ylabel(r"$\rho$ (veh/m)")
ax.set_title("Shock: density profiles sliding upstream (red light)"); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig("shock_profiles.png", dpi=110); plt.show()

# ---------------- RAREFACTION (green light) + mass ledger ----------------
rho = np.where(x < x0, RJAM, 0.0).astype(float)
RST = []; mass0 = rho.sum()*dx; outflux = 0.0
for k in range(steps+1):
    if k % se == 0: RST.append(rho.copy())
    if k < steps:
        rho, F = godunov_step(rho, dt, dx, rho[0], 0.0)
        outflux += (F[-1]-F[0])*dt
ledger = abs((mass0 - rho.sum()*dx) - outflux)
print(f"[RAREF] mass ledger |dmass - net flux| = {ledger:.3e} veh (machine precision)")

RST = np.array(RST)
fig, ax = plt.subplots(figsize=(6,4.5))
for fr in [0.0,0.2,0.5,1.0]:
    kk = int(fr*(len(RST)-1)); ax.plot(x, RST[kk], lw=1.6, label=f"t={kk*se*dt:.0f} s")
ax.set_xlabel("x (m)"); ax.set_ylabel(r"$\rho$ (veh/m)")
ax.set_title("Rarefaction: smooth spreading fan (green light)"); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig("rarefaction_profiles.png", dpi=110); plt.show()

print(f"\nparams: vf={VF:.3f} m/s ({VF*3.6:.0f} km/h), rjam={RJAM}, qmax={QMAX:.4f} veh/s, "
      f"upstream load={rL/RJAM:.2f}, dx={dx} m, dt={dt:.3f} s, steps={steps}")